# Task 3 - Agentic Workflows: Multi-Agent Financial Research System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/CDAZZDEV-MLE-YourName/blob/main/task3_agentic/notebooks/Task3_Agentic_Workflows.ipynb)

Built with **LangGraph** + a Groq tool-calling model. CPU-only - no GPU needed.

**Before running:** add `GROQ_API_KEY` as a Colab secret (key icon, left sidebar) or set it in the cell below.

In [46]:
# Uncomment for a fresh Colab environment
# !pip install -q -r ../requirements.txt

import sys, os
sys.path.append(os.path.abspath(".."))


In [47]:
import os
try:
    from google.colab import userdata
    os.environ.setdefault("GROQ_API_KEY", userdata.get("GROQ_API_KEY") or "")
except Exception:
    pass

# The smaller model leaves room for this workflow's multiple tool and structuring calls.
os.environ["AGENT_MODEL"] = "openai/gpt-oss-20b"
os.environ["AGENT_MAX_TOKENS"] = "600"
os.environ["STRUCTURING_MAX_TOKENS"] = "400"
os.environ["AGENT_MAX_RETRIES"] = "0"
os.environ["AGENT_RECURSION_LIMIT"] = "12"

TICKER = "AAPL"  # change to any ticker of your choice
print(f"Agent model: {os.environ['AGENT_MODEL']}")


Agent model: openai/gpt-oss-20b


## Task 3A - Single Tool-Using Research Agent

In [48]:
from src.single_agent import build_agent, run_query

agent = build_agent()
result = run_query(agent, TICKER)
print(result["report"])


/Users/chiransiriwardena/Documents/Financial-AI/task3_agentic/src/single_agent.py:72: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, ALL_TOOLS, checkpointer=checkpointer, prompt=SYSTEM_PROMPT)
INFO:src.memory:Persistent cache HIT for AAPL at /Users/chiransiriwardena/Documents/Financial-AI/task3_agentic/cache/AAPL_2026-09-13.json - skipping tool/LLM calls.


**## Financial Health Summary**  
- **Current price:** $332.27 (6‑month snapshot) – trading ~4 % below the 52‑week high of $344.57, but comfortably above the 52‑week low of $245.51.  
- **Momentum indicators:** 50‑day SMA = $317.97 (price 4.5 % above SMA) and RSI = 62.8, indicating the stock is in a modestly bullish zone but not yet overbought.  
- **Historical volatility:** 31.97 % annualized (30‑day window) – higher than the S&P 500’s ~18 % level, reflecting the “Magnificent 7” premium and sensitivity to macro news.  
- **Market sentiment:** LLM‑scored sentiment on the 10 most recent headlines is **positive** (overall score 0.4). The majority of headlines highlight buying opportunities and strong growth themes (e.g., “Apple, Taiwan Semi Lead Five Stocks Near Buy Points”; “Dow Jones Futures … Apple … Are New Buys”).  

Overall, Apple shows solid price strength, healthy momentum, and a favorable short‑term sentiment backdrop, but the elevated volatility signals that the share price can

### Task 3C - Short-term memory demonstration
Ask a follow-up on the SAME thread - the agent should answer from what it already
retrieved above, without necessarily calling a tool again. Check the tool-call count
in `agent_trace.jsonl` before/after to confirm.

In [49]:
import json

with open("../logs/agent_trace.jsonl") as f:
    calls_before = sum(1 for _ in f)

from src.single_agent import ask_followup
followup_answer = ask_followup(agent, f"Based on what you already found, is {TICKER}'s RSI closer to overbought or oversold territory?")
print(followup_answer)

with open("../logs/agent_trace.jsonl") as f:
    calls_after = sum(1 for _ in f)

print(f"\nTool calls logged before follow-up: {calls_before}, after: {calls_after}")
print("(if these are equal, the follow-up was answered purely from short-term memory)")


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



=== Task 3A follow-up (short-term memory) ===
[human] Based on what you already found, is AAPL's RSI closer to overbought or oversold territory?
[ai] 
    -> tool_call: get_price_data({'period': '1y', 'ticker': 'AAPL'})
[tool] {"ticker": "AAPL", "period": "1y", "current_price": 332.2699890136719, "fifty_two_week_high": 344.57000732421875, "fifty_two_week_low": 229.02000427246094, "sma_50": 317.97419982910156, "sma_200": 284.9533494567871, "rsi_14": 62.7890096388532, "num_trading_days": 251}
[ai] AAPL’s 14‑day RSI is **62.8** (from the 1‑year price data).  
With the typical overbought threshold at 70 and oversold at 30, the RSI is **closer to overbought territory**—though it is still below the conventional overbought level.
AAPL’s 14‑day RSI is **62.8** (from the 1‑year price data).  
With the typical overbought threshold at 70 and oversold at 30, the RSI is **closer to overbought territory**—though it is still below the conventional overbought level.

Tool calls logged before follow-u

### Task 3C - Persistent memory demonstration
Running the same ticker again (same day) should hit the cache and skip the agent
entirely - notice no new lines get appended to `agent_trace.jsonl`.

In [50]:
with open("../logs/agent_trace.jsonl") as f:
    calls_before_cache_run = sum(1 for _ in f)

cached_result = run_query(agent, TICKER)  # same ticker, same day -> cache hit
print(cached_result["report"][:300], "...")

with open("../logs/agent_trace.jsonl") as f:
    calls_after_cache_run = sum(1 for _ in f)

print(f"\nTool calls before: {calls_before_cache_run}, after: {calls_after_cache_run} (should be equal on a cache hit)")


INFO:src.memory:Persistent cache HIT for AAPL at /Users/chiransiriwardena/Documents/Financial-AI/task3_agentic/cache/AAPL_2026-09-13.json - skipping tool/LLM calls.


**## Financial Health Summary**  
- **Current price:** $332.27 (6‑month snapshot) – trading ~4 % below the 52‑week high of $344.57, but comfortably above the 52‑week low of $245.51.  
- **Momentum indicators:** 50‑day SMA = $317.97 (price 4.5 % above SMA) and RSI = 62.8, indicating the stock is in a ...

Tool calls before: 73, after: 73 (should be equal on a cache hit)


### Task 3C - Observability
Every tool call above was logged to `agent_trace.jsonl` with inputs, a truncated
output, and duration.

In [51]:
import pandas as pd

trace_records = [json.loads(l) for l in open("../logs/agent_trace.jsonl")]
pd.DataFrame(trace_records)


/var/folders/zs/_zj9907x6cq8c14rbmv6cqs80000gn/T/ipykernel_68097/2370450794.py:3: ResourceWarning: unclosed file <_io.TextIOWrapper name='../logs/agent_trace.jsonl' mode='r' encoding='UTF-8'>
  trace_records = [json.loads(l) for l in open("../logs/agent_trace.jsonl")]


,timestamp,tool,inputs,output,error,duration_seconds
0,2026-09-13T05:45:32.652212+00:00,get_price_data,"{'ticker': 'AAPL', 'period': '6mo'}","{""ticker"": ""AAPL"", ""period"": ""6mo"", ""current_p...",None,1.2830
1,2026-09-13T05:45:33.891520+00:00,get_news,"{'ticker': 'AAPL', 'n': 10}","{""ticker"": ""AAPL"", ""headlines"": [{""headline"": ...",None,0.5920
2,2026-09-13T05:45:35.232311+00:00,calculate_volatility,"{'ticker': 'AAPL', 'window': 30}","{""ticker"": ""AAPL"", ""window_days"": 30, ""daily_s...",None,0.5022
3,2026-09-13T05:45:38.314736+00:00,llm_sentiment,{'headlines': ['Apple's AI vision is tailored ...,"{""headline_count"": 10, ""overall_score"": 0.4, ""...",None,1.6361
4,2026-09-13T05:45:39.700771+00:00,web_search,{'query': 'Apple supply chain risk Taiwan semi...,"{""error"": ""duckduckgo-search is not installed....",None,0.0006
...,...,...,...,...,...,...
68,2026-09-13T06:10:36.595254+00:00,calculate_volatility,"{'ticker': 'AAPL', 'window': 30}","{""ticker"": ""AAPL"", ""window_days"": 30, ""daily_s...",None,0.4646
69,2026-09-13T06:10:41.104413+00:00,get_price_data,"{'ticker': 'AAPL', 'period': '1y'}","{""ticker"": ""AAPL"", ""period"": ""1y"", ""current_pr...",None,0.4897
70,2026-09-13T06:10:43.194670+00:00,get_news,"{'ticker': 'AAPL', 'n': 10}","{""ticker"": ""AAPL"", ""headlines"": [{""headline"": ...",None,0.7094
71,2026-09-13T06:10:44.663185+00:00,web_search,{'query': 'Apple 2024 supply chain issues Q3 2...,"{""error"": ""No web search results for query='Ap...",None,0.7102


## Task 3B - Multi-Agent Coordination

In [52]:
import importlib
import src.config
import src.multi_agent

importlib.reload(src.config)
importlib.reload(src.multi_agent)
from src.multi_agent import run_multi_agent_pipeline

multi_result = run_multi_agent_pipeline(TICKER)

print("=== Agent A's Data Brief ===")
print(multi_result["data_brief"].model_dump_json(indent=2))

print("\n=== Agent B's Clarification Request ===")
print(multi_result["clarification_request"].question)

print("\n=== Agent A's Clarification Response ===")
print(multi_result["clarification_response"].answer)

print("\n=== Agent B's Final Structured Report ===")
print(multi_result["final_report"].model_dump_json(indent=2))


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



=== Agent A -> initial brief (AAPL) ===
[human] Produce a quantitative data brief for AAPL: current price and moving averages / RSI from get_price_data, and 30-day annualized volatility from calculate_volatility. You do not have news headlines yet, so leave sentiment as neutral (0.0) with a note that it is pending Agent B's headlines.
[ai] 
    -> tool_call: get_price_data({'ticker': 'AAPL'})
[tool] {"ticker": "AAPL", "period": "1y", "current_price": 332.2699890136719, "fifty_two_week_high": 344.57000732421875, "fifty_two_week_low": 229.02000427246094, "sma_50": 317.97419982910156, "sma_200": 284.9533494567871, "rsi_14": 62.7890096388532, "num_trading_days": 251}
[ai] 
    -> tool_call: calculate_volatility({'ticker': 'AAPL', 'window': 30})
[tool] {"ticker": "AAPL", "window_days": 30, "daily_std": 0.02013988734307453, "annualized_volatility_pct": 31.97}
[ai] **Quantitative Data Brief – Apple Inc. (AAPL)**  

| Metric | Value | Notes |
|--------|-------|-------|
| **Current price** | *

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



=== Agent B -> clarification request (AAPL) ===
[human] You are drafting a research report for AAPL and have received this
quantitative brief from Agent A (the Data Analyst):

{
  "ticker": "AAPL",
  "current_price": 332.27,
  "annualized_volatility_pct": 31.97,
  "sentiment_score": 0.0,
  "sentiment_label": "neutral",
  "sma_50": 317.97,
  "sma_200": 28
[ai] Agent A, could you provide the consensus price target for AAPL for the next 12 months?


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



=== Agent A -> clarification response (AAPL) ===
[human] Agent B (the Research Writer) has a follow-up question about AAPL:

"Agent A, could you provide the consensus price target for AAPL for the next 12 months?"

Your original brief was:
{
  "ticker": "AAPL",
  "current_price": 332.27,
  "annualized_volatility_pct": 31.97,
  "sentiment_score": 0.0,
  "s
[ai] 
    -> tool_call: get_price_data({'period': '1y', 'ticker': 'AAPL'})
[tool] {"ticker": "AAPL", "period": "1y", "current_price": 332.2699890136719, "fifty_two_week_high": 344.57000732421875, "fifty_two_week_low": 229.02000427246094, "sma_50": 317.97419982910156, "sma_200": 284.9533494567871, "rsi_14": 62.7890096388532, "num_trading_days": 251}
[ai] **Consensus price‑target for AAPL (next 12 months)**  
I do not have access to analyst consensus data, so I cannot quote an official target.  
Using the quantitative data available:

| Metric | Value |
|--------|-------|



INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
/Users/chiransiriwardena/Documents/Financial-AI/task3_agentic/src/tools.py:240: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
INFO:primp:response: https://www.bing.com/search?q=Apple+AAPL+analyst+price+target+2026 200
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"


=== Agent A's Data Brief ===
{
  "ticker": "AAPL",
  "current_price": 332.27,
  "annualized_volatility_pct": 31.97,
  "sentiment_score": 0.0,
  "sentiment_label": "neutral",
  "sma_50": 317.97,
  "sma_200": 284.95,
  "rsi_14": 62.79,
  "fifty_two_week_high": 344.57,
  "fifty_two_week_low": 229.02,
  "notes": null
}

=== Agent B's Clarification Request ===
Agent A, could you provide the consensus price target for AAPL for the next 12 months?

=== Agent A's Clarification Response ===
**Consensus price‑target for AAPL (next 12 months)**  
I do not have access to analyst consensus data, so I cannot quote an official target.  
Using the quantitative data available:

| Metric | Value |
|--------|-------|


=== Agent B's Final Structured Report ===
{
  "ticker": "AAPL",
  "financial_health_summary": "AAPL is trading at $332.27; 30-day annualized volatility is 31.97%. The qualitative search stage was unavailable because the provider rate limit was reached.",
  "top_risks": [
    {
      "risk"

The full agent-by-agent message trace (what each agent said, which tools it
called, and the timing) was printed above via `print_trace()` and is also saved to
`logs/agent_message_trace.jsonl` for the record.

In [53]:
message_records = [json.loads(l) for l in open("../logs/agent_message_trace.jsonl")]
pd.DataFrame(message_records).tail(20)

/var/folders/zs/_zj9907x6cq8c14rbmv6cqs80000gn/T/ipykernel_68097/2034272660.py:1: ResourceWarning: unclosed file <_io.TextIOWrapper name='../logs/agent_message_trace.jsonl' mode='r' encoding='UTF-8'>
  message_records = [json.loads(l) for l in open("../logs/agent_message_trace.jsonl")]


,timestamp,label,role,content,tool_calls
152,2026-09-13T06:10:41.820554+00:00,Agent A -> clarification response (AAPL),human,Agent B (the Research Writer) has a follow-up ...,None
153,2026-09-13T06:10:41.821215+00:00,Agent A -> clarification response (AAPL),ai,,"[{'name': 'get_price_data', 'args': {'period':..."
154,2026-09-13T06:10:41.821565+00:00,Agent A -> clarification response (AAPL),tool,"{""ticker"": ""AAPL"", ""period"": ""1y"", ""current_pr...",None
155,2026-09-13T06:10:41.821802+00:00,Agent A -> clarification response (AAPL),ai,**AAPL 200‑day Simple Moving Average (SMA‑200)...,None
156,2026-09-13T06:12:48.898499+00:00,Task 3A follow-up (short-term memory),human,"Based on what you already found, is AAPL's RSI...",None
157,2026-09-13T06:12:48.900111+00:00,Task 3A follow-up (short-term memory),ai,,"[{'name': 'get_price_data', 'args': {'period':..."
158,2026-09-13T06:12:48.900270+00:00,Task 3A follow-up (short-term memory),tool,"{""ticker"": ""AAPL"", ""period"": ""1y"", ""current_pr...",None
159,2026-09-13T06:12:48.900363+00:00,Task 3A follow-up (short-term memory),ai,AAPL’s 14‑day RSI is **62.8** (from the 1‑year...,None
160,2026-09-13T06:12:54.983167+00:00,Agent A -> initial brief (AAPL),human,Produce a quantitative data brief for AAPL: cu...,None
161,2026-09-13T06:12:54.983698+00:00,Agent A -> initial brief (AAPL),ai,,"[{'name': 'get_price_data', 'args': {'ticker':..."


## Bonus - Observability Dashboard
Run this from a terminal (not this notebook cell) after the runs above have populated the logs:

In [54]:
# !streamlit run ../src/dashboard.py
print("Run: streamlit run ../src/dashboard.py")


Run: streamlit run ../src/dashboard.py


## Summary

- **Task 3A**: a single LangGraph ReAct agent with all five tools, deciding
  autonomously which to call based on observed results, producing a three-section
  evidence-backed report, with graceful fallback on tool errors.
- **Task 3B**: two role-restricted agents (Data Analyst / Research Writer) handing
  off a validated Pydantic `DataBrief`, completing one clarification critique loop,
  and producing a final validated `ResearchReport` end-to-end with no manual steps.
- **Task 3C**: short-term memory via a reused LangGraph `thread_id`, persistent
  memory via a ticker+date JSON cache, and full tool-call observability via
  `agent_trace.jsonl`.
- **Bonus**: a Streamlit dashboard visualizing the trace data.

See `../README.md` for the full requirement-to-code mapping and `../CITATIONS.md` /
`../REFLECTION.md` for AI-usage disclosure and architectural reflection.